# QQQ Direction Backtest — LIVE API (2026-01-01..2026-08-30, N=50)

Builds predictions **from scratch via the live TypeSafe API** — it **never reads** `backtest_predictions.jsonl` and **writes no cache file**; results stay in memory for the visualizations.

**Prereqs:** `export TYPESAFE_API_KEY="..."` before starting the kernel; network to TypeSafe + `yfinance`.
Expect ~165 live predictions at `WORKERS=20` (a few minutes). Every re-run makes fresh calls (no resume cache).

Run top-to-bottom: `Kernel → Restart & Run All`.

In [ ]:
import os
if not os.environ.get('TYPESAFE_API_KEY', '').strip():
    raise RuntimeError('TYPESAFE_API_KEY env var is not set — export it, then restart the kernel.')

START = '2026-01-01'
END = '2026-08-30'
N = 50            # observation window (target key N+1 = 51)
WORKERS = 20      # parallel prediction threads (allowed 1-64)

import datetime
import concurrent.futures
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from predict_qqq_direction import build_state, compute_log_pct_change, predict_direction, validate_num_timepoints
from backtest_qqq_direction import compute_metrics, direction_of, fetch_range_closes, validate_workers

validate_num_timepoints(N); validate_workers(WORKERS)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print(f'live backtest: {START}..{END}  N={N}  workers={WORKERS}')

## 1. Fetch QQQ history (with warmup buffer) — no cached predictions involved

In [ ]:
start_d = datetime.date.fromisoformat(START)
end_d = datetime.date.fromisoformat(END)
fetch_start = (start_d - datetime.timedelta(days=max(400, int((N + 5) * 1.4)))).isoformat()
fetch_end = (end_d + datetime.timedelta(days=1)).isoformat()
closes, dates = fetch_range_closes(fetch_start, fetch_end)
changes = compute_log_pct_change(closes)
eligible = [j for j, d in enumerate(dates) if start_d <= d <= end_d and j >= N + 1]
warmup_skipped = sum(1 for j, d in enumerate(dates) if start_d <= d <= end_d and j < N + 1)
print(f'closes: {len(closes)}  eligible target days: {len(eligible)}  warmup-skipped: {warmup_skipped}')

## 2. LIVE predictions via TypeSafe API (in-memory only, nothing written to disk)

In [ ]:
def predict_one(target_date, state, actual_change, actual, n):
    try:
        result = predict_direction(state)  # client-default model, same path as predict_qqq_direction.py
        predicted = result['prediction']
        return {'date': target_date.isoformat(), 'predicted': predicted,
                'confidence': float(result['confidence']), 'actual': actual,
                'actual_change_pct': round(float(actual_change), 4),
                'correct': predicted == actual, 'timepoints': n}
    except Exception as exc:
        return {'date': target_date.isoformat(), 'error': str(exc)}

print(f'{len(eligible)} live calls to make')

tasks = []
for j in eligible:  # states pre-built on main thread (pure + deterministic)
    tasks.append((dates[j], build_state(changes[j - N - 1:j - 1], N), changes[j - 1], direction_of(changes[j - 1])))

records = []
done = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futs = {pool.submit(predict_one, d, s, ac, a, N): d for d, s, ac, a in tasks}
    for fut in concurrent.futures.as_completed(futs):
        records.append(fut.result())
        done += 1
        if done % 25 == 0 or done == len(tasks):
            print(f'live: {done}/{len(tasks)} predictions done')

metrics = compute_metrics(records)
n_err = sum(1 for r in records if 'error' in r)
print(f"predicted days: {metrics['total']}  accuracy: {metrics['accuracy']:.4f}  "
      f"baseline: {metrics['baseline']:.4f}  errors skipped: {n_err}")

## 3. Frame predictions + merge with real QQQ closes

In [ ]:
import yfinance as yf

df = pd.DataFrame([r for r in records if 'error' not in r])
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date').sort_index()
df['pred_label'] = df['predicted']
acc = df['correct'].mean()
print(f"success: {len(df)}  |  UP preds: {(df['pred_label']=='UP').sum()}  |  DOWN preds: {(df['pred_label']=='DOWN').sum()}  |  acc: {acc:.4f}")

px = yf.download('QQQ', start=(df.index.min() - pd.Timedelta(days=10)).strftime('%Y-%m-%d'),
                 end=(df.index.max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
                 interval='1d', auto_adjust=True, progress=False)
if hasattr(px.columns, 'droplevel'):
    try:
        px.columns = px.columns.droplevel(1)
    except (ValueError, IndexError, KeyError, TypeError):
        pass
col = 'Close' if 'Close' in px.columns else 'Adj Close'
price = px[[col]].rename(columns={col: 'Close'}).copy()
price.index = pd.to_datetime(price.index).tz_localize(None)
m = pd.merge(price.dropna(), df, left_index=True, right_index=True, how='inner')
print(f'merged rows: {len(m)} / {len(df)}')
m[['Close', 'pred_label', 'actual', 'confidence', 'correct']].head(3)

## 4. ⭐ Main chart — QQQ price with prediction regimes (green=UP / red=DOWN, alpha=0.3)

In [ ]:
def add_prediction_background(ax, dates, preds, alpha=0.3):
    """Shade contiguous prediction regimes: UP=green, DOWN=red. Returns run count."""
    assert len(dates) == len(preds)
    runs, s = [], 0
    for i in range(1, len(preds) + 1):
        if i == len(preds) or preds[i] != preds[s]:
            runs.append((dates[s], dates[i-1] + pd.Timedelta(days=1), preds[s]))
            s = i
    for d0, d1, lab in runs:
        ax.axvspan(d0, d1, color='green' if lab == 'UP' else 'red', alpha=alpha, linewidth=0)
    return len(runs)

fig, ax = plt.subplots(figsize=(14, 5))
n_runs = add_prediction_background(ax, m.index, m['pred_label'].values, alpha=0.3)
ax.plot(m.index, m['Close'], color='#1f3a93', lw=1.5, zorder=3, label='QQQ Close')
ax.set_title(f'QQQ Price with LIVE Direction Predictions 2026 ({len(m)} days, {n_runs} regimes)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price ($)')
ax.legend(handles=[mpatches.Patch(color='green', alpha=0.3, label='Predicted UP'),
                   mpatches.Patch(color='red', alpha=0.3, label='Predicted DOWN'),
                   plt.Line2D([], [], color='#1f3a93', label='QQQ Close')], loc='upper left')
fig.tight_layout(); plt.show()

### 4b. Zoomed view — last 60 days (regime detail + correctness dots)

In [ ]:
zoom = m.loc[m.index >= m.index.max() - pd.Timedelta(days=60)]
fig, ax = plt.subplots(figsize=(14, 4.5))
add_prediction_background(ax, zoom.index, zoom['pred_label'].values, alpha=0.3)
ax.plot(zoom.index, zoom['Close'], color='#1f3a93', lw=1.8, zorder=3)
ok, bad = zoom[zoom['correct']], zoom[~zoom['correct']]
ax.scatter(ok.index, ok['Close'], s=18, color='lime', edgecolors='darkgreen', linewidths=0.6, zorder=4, label=f'correct ({len(ok)})')
ax.scatter(bad.index, bad['Close'], s=22, color='black', marker='x', linewidths=1.0, zorder=4, label=f'wrong ({len(bad)})')
ax.set_title('QQQ — last 60 days: live prediction regimes + correctness', fontweight='bold')
ax.set_ylabel('Price ($)'); ax.legend(loc='upper left', fontsize=9)
fig.tight_layout(); plt.show()

## 5. Strategy equity — long/short vs long/flat vs buy & hold
*Assumption (documented): `actual_change_pct` is log-return×100. Long/short takes +r on UP preds, −r on DOWN preds; long/flat takes +r on UP, 0 on DOWN.*

In [ ]:
m['logret'] = m['actual_change_pct'] / 100.0
m['strat_ls'] = np.where(m['pred_label'] == 'UP', m['logret'], -m['logret'])
m['strat_lf'] = np.where(m['pred_label'] == 'UP', m['logret'], 0.0)
for c in ['logret', 'strat_ls', 'strat_lf']:
    m[f'eq_{c}'] = np.exp(m[c].cumsum())

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(m.index, m['eq_logret'], label='Buy & Hold', color='black', lw=1.6)
ax.plot(m.index, m['eq_strat_ls'], label='Strategy long/short', color='#0a7d2c', lw=1.4)
ax.plot(m.index, m['eq_strat_lf'], label='Strategy long/flat', color='#2ca02c', lw=1.2, ls='--')
ax.fill_between(m.index, m['eq_strat_ls'], m['eq_logret'], where=m['eq_strat_ls'] >= m['eq_logret'],
                color='green', alpha=0.12, interpolate=True)
ax.fill_between(m.index, m['eq_strat_ls'], m['eq_logret'], where=m['eq_strat_ls'] < m['eq_logret'],
                color='red', alpha=0.12, interpolate=True)
ax.set_title('Equity curves (rebased 1.0) — live strategy vs buy & hold', fontweight='bold')
ax.set_ylabel('Growth of $1'); ax.legend()
fig.tight_layout(); plt.show()
print(m[[c for c in m.columns if c.startswith('eq_')]].iloc[-1].round(3).to_string())

## 6. Drawdown + rolling accuracy (20-day)

In [ ]:
eq = m['eq_strat_ls']
dd = eq / eq.cummax() - 1.0
roll_acc = m['correct'].rolling(20, min_periods=5).mean()

fig, (a1, a2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True, gridspec_kw={'height_ratios': [1.4, 1]})
a1.plot(m.index, eq, color='#0a7d2c', lw=1.4)
a1.fill_between(m.index, dd * eq.cummax() + eq.cummax() * 0, eq, color='red', alpha=0.15)
a1.set_title('Strategy drawdown (long/short equity vs running peak fill)', fontweight='bold')
a1.set_ylabel('Equity'); a1.text(0.01, 0.92, f'max DD: {dd.min():.1%}', transform=a1.transAxes, fontsize=11,
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
a2.plot(roll_acc.index, roll_acc, color='#6a1b9a', lw=1.4, label='20-day rolling accuracy')
a2.axhline(0.5, color='gray', ls='--', lw=1, label='coin flip')
a2.axhline(acc, color='red', ls=':', lw=1.2, label=f'overall {acc:.3f}')
a2.set_ylim(0, 1); a2.set_ylabel('Accuracy'); a2.legend(fontsize=9)
fig.tight_layout(); plt.show()

## 7. Confusion matrix + confidence diagnostics

In [ ]:
ct = pd.crosstab(m['pred_label'], m['actual'], rownames=['Predicted'], colnames=['Actual'])
ct = ct.reindex(index=['UP', 'DOWN'], columns=['UP', 'DOWN'], fill_value=0)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
im = a1.imshow(ct.values, cmap='Blues')
a1.set_xticks([0, 1], ['UP', 'DOWN']); a1.set_yticks([0, 1], ['UP', 'DOWN'])
a1.set_xlabel('Actual'); a1.set_ylabel('Predicted')
a1.set_title('Confusion matrix (counts)', fontweight='bold')
for i in range(2):
    for j in range(2):
        a1.text(j, i, ct.values[i, j], ha='center', va='center', fontsize=14, fontweight='bold')
fig.colorbar(im, ax=a1, shrink=0.8)
a2.hist(m[m['correct']]['confidence'], bins=30, alpha=0.6, label='correct', color='green')
a2.hist(m[~m['correct']]['confidence'], bins=30, alpha=0.5, label='wrong', color='red')
a2.set_xlabel('Confidence'); a2.set_ylabel('Days'); a2.legend()
a2.set_title('Confidence distribution: correct vs wrong', fontweight='bold')
fig.tight_layout(); plt.show()
print(ct.to_string())
print(m.groupby(pd.cut(m['confidence'], [0, 0.6, 0.8, 1.01]))['correct'].agg(['mean', 'count']).round(4).to_string())

## 8. Return separation + monthly accuracy (2026)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
up_r = m[m['pred_label']=='UP']['actual_change_pct']
dn_r = m[m['pred_label']=='DOWN']['actual_change_pct']
a1.hist(dn_r, bins=40, alpha=0.55, color='red', label=f'DOWN pred (n={len(dn_r)})')
a1.hist(up_r, bins=30, alpha=0.65, color='green', label=f'UP pred (n={len(up_r)})')
if len(up_r): a1.axvline(up_r.median(), color='darkgreen', ls='--', label=f'UP median {up_r.median():+.2f}%')
if len(dn_r): a1.axvline(dn_r.median(), color='darkred', ls='--', label=f'DOWN median {dn_r.median():+.2f}%')
a1.set_xlim(-6, 6); a1.set_xlabel('Realized next-day change %'); a1.set_ylabel('Days')
a1.set_title('Do UP calls earn higher realized returns?', fontweight='bold')
a1.legend(fontsize=8)
mo = m.assign(month=m.index.strftime('%Y-%m')).groupby('month')['correct'].agg(['mean', 'count'])
a2.bar(mo.index, mo['mean'], color=np.where(mo['mean'] >= 0.5, 'green', 'red'), alpha=0.7)
a2.axhline(0.5, color='gray', ls='--'); a2.axhline(acc, color='black', ls=':', label=f'overall {acc:.3f}')
a2.set_ylim(0, 1); a2.set_ylabel('Accuracy'); a2.set_title('Accuracy by month (2026)', fontweight='bold')
plt.setp(a2.get_xticklabels(), rotation=45, ha='right'); a2.legend(fontsize=9)
fig.tight_layout(); plt.show()